<a href="https://colab.research.google.com/github/AbdullahRasheed452/ML-Internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess

REPO_URL = "https://github.com/AbdullahRasheed452/ML-Internship"
REPO_DIR = "ML-Internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if os.getcwd().split("/")[-1] != REPO_DIR:
    os.chdir(REPO_DIR)

In [2]:
%pip -q install -U duckdb huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 111.7 MB/s eta 0:00:00


In [3]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('HF_TOKEN')

In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

## 1. Question

*The research question and the decision it supports.*

In [5]:
# Question: Which content pages should a reviewer check first for
# refresh or optimization, based on real search performance data
# Decision this supports: prioritizing limited reviewer time toward
# the pages most likely to be declining or underperforming

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [6]:
# Data used: FlyRank internship warehouse release on Hugging Face
# Tables: fact_content_daily_performance, dim_content, dim_clients
# Time window: March 2026, a full month, kept away from the sealed final month (June 2026)
# Excluded on purpose: health_score, priority_score, action_type,
# since these are the app's own decisions, not raw observed data
# All ids are pseudonymized, no client names, URLs, or raw queries are used anywhere in this work

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [7]:
# Assumptions: a page is treated as declining if its impressions
# dropped more than 20 percent from the first half of March to the second half

# Features used, all built only from the first half of March so
# nothing from the label window leaks in:
# avg_impressions_first_half, avg_position_first_half,
# avg_ctr_first_half, word_count, content_age_days

# Label: is_declining, based on the impression drop described above

# Baseline: a simple rule comparing each page's CTR to the normal
# CTR for its position tier, the bigger the gap, the higher the score

# Validation: split by client, so the same client never appears in
# both train and test, this avoids the model memorizing one client's
# pattern instead of learning something general

# Leakage check: an earlier version of the impressions feature used
# the whole month, including the second half the label comes from,
# which gave a fake near perfect score, this was caught and fixed by switching to first half only features

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [8]:
# Results on the same client held out test split

# Precision@20: baseline 0.300, model 0.650
# Precision@50: baseline 0.260, model 0.700

# The model beats the baseline at both cutoffs on this test split
# This is an observed, directional result on one split of one month
# of data, not a guarantee it will always outperform the baseline

## 5. Limitations

*What this work cannot claim.*

In [9]:
# This work cannot claim:
# That fixing a flagged page will guarantee a traffic recovery
# That the model works the same way on other months or new clients not seen in this data
# Any knowledge of how Google's search ranking algorithm actually works
# That the ranked queue is a measure of content quality, only of arisk pattern found in this sample

# Real limits of the data itself:
# Clients have unequal history, so trends are not equally reliable across every client
# Only a small share of rows have GA4 engagement data, most rows are search only

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [10]:
# Ranked recommendations, from the Week 7 action playbook

# Reason codes and matching actions:
# low_ctr_for_position: good rank but low clicks, review title and meta
# aging_content_risk: over 270 days old, schedule a refresh
# visible_page_declining: real traffic and high risk, prioritize this week
# general_decline_risk: lower confidence, just monitor

# Most flagged pages fall under low_ctr_for_position, meaning they
# already have visibility but are not converting it into clicks

# What should never be automated:
# Auto deleting or unpublishing a page based on this score
# Auto rewriting content without a human reviewing it first
# Making client facing promises based on this score alone

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [11]:
# Collecting the figures and tables the deployed paper will show
import json

with open("work/outputs/model_metrics.json") as f:
    metrics = json.load(f)

print(metrics)

# The paper will embed:
# work/figures/reason_code_counts.png, the reason code bar chart
# The results table above, model vs baseline precision numbers
# The ranked recommendations table from Section 6

{'model': 'Random Forest', 'split': 'grouped by client', 'precision_at_20': 0.65, 'precision_at_50': 0.7, 'baseline_precision_at_20': 0.3, 'baseline_precision_at_50': 0.26, 'total_pages_ranked': 17925, 'month_used': '2026-03'}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.